In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 21. RLHF — text text textTraining ⭐ [CPU/GPU Benchmark ⑨]

> **Learning Goals**
> - textTraining text text(MDP, text, text, text)text LLMtext text
> - text text(Policy Gradient)text REINFORCE text textdegreestext
> - PPOtext clipped objectivetext text text
> - text Model(RM) Trainingtext text

## 21.1 SFTtext text text Problem

SFT Modeltext "text text"text text:
- text **text** text text
- text textdegrees text text text text
- "text(alignment)" text: Modeltext text text text

## 21.2 RLHF 3text

1. **SFT**: text text (Ch 20)
2. **Reward Model (RM)**: text text text text Model Training
3. **PPO**: RMtext text text(=LLM) textTraining

## 21.3 textTraining text — MDP

**MDP** (Markov Decision Process): $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$
- $\mathcal{S}$: text Space
- $\mathcal{A}$: text Space
- $P(s'|s, a)$: text text
- $R(s, a)$: text
- $\gamma$: text

LLM text:
- text $s_t$ = text text text $[w_0, \ldots, w_{t-1}]$
- text $a_t$ = text text $w_t$
- text $\pi_\theta(a_t | s_t) = P(w_t | w_{<t}; \theta)$
- text $R$ = RMtext text text text (text text text)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# text text (toy)
vocab_size = 100  # text text text text
print(f"vocab_size (toy): {vocab_size}")

## 21.4 text text text (REINFORCE)

text: $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$ text.

$$\nabla J(\theta) = \mathbb{E}_{\tau} \left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

- $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$: text text text
- $\nabla \log \pi$: text text text → "text text text text text" text
- $G_t$text text → text text text text text text

**Advantage** $A_t = G_t - V(s_t)$: text $V$ text text text.


In [ ]:
# REINFORCE text (text toy MDP)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# text: text bandit (3text text, text Distribution text)
true_rewards = [0.2, 0.5, 0.8]  # text 2text text

class PolicyNet(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(1, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, x):
        return self.head(F.relu(self.fc(x)))

policy = PolicyNet(3)
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

n_episodes = 500
all_rewards = []
for ep in range(n_episodes):
    # text: 1 step (bandit)
    state = torch.tensor([[1.0]])
    logits = policy(state)
    probs = F.softmax(logits, dim=-1)
    action = torch.multinomial(probs, 1).item()
    reward = true_rewards[action] + np.random.randn() * 0.05

    # REINFORCE: log_prob * reward
    log_prob = F.log_softmax(logits, dim=-1)[0, action]
    loss = -log_prob * reward  # text → text text
    opt.zero_grad()
    loss.backward()
    opt.step()
    all_rewards.append(reward)

print(f"text 50 text Mean Reward: {np.mean(all_rewards[:50]):.3f}")
print(f"text 50 text Mean Reward: {np.mean(all_rewards[-50:]):.3f}")
print(f"Theory Maximum: {max(true_rewards):.3f}")


## 21.5 PPO — Trust Regiontext Clipped Objective

REINFORCEtext text text. PPOtext text text text text text text text.

**text**: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

**Clipped Objective**:
$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[\min\left(r_t \hat{A}_t, \, \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

- $\epsilon \approx 0.2$: text text text
- mintext clip → text text text text text text
- trust regiontext text


In [ ]:
# PPO text (toy Problem)
import torch
import torch.nn as nn
import torch.nn.functional as F

class Policy(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(4, 32)
        self.head = nn.Linear(32, n_actions)
    def forward(self, x):
        return self.head(F.tanh(self.fc(x)))

# text text (text)
def env_step(state, action):
    reward = float((state.sum() + action - 2) / 10)  # text text
    next_state = state + torch.randn(4) * 0.1
    return next_state, reward, False

policy = Policy(3)
opt = torch.optim.Adam(policy.parameters(), lr=3e-4)

# PPO text
def ppo_update(policy, opt, states, actions, old_log_probs, advantages, returns, values,
               epsilon=0.2, n_epochs=4):
    for _ in range(n_epochs):
        logits = policy(states)
        new_log_probs = F.log_softmax(logits, dim=-1)
        new_log_probs_a = new_log_probs.gather(1, actions.unsqueeze(1)).squeeze()

        ratio = torch.exp(new_log_probs_a - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        # text Function text (text)
        critic_loss = F.mse_loss(values.squeeze(), returns)

        loss = actor_loss + 0.5 * critic_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

# Training text
n_iterations = 50
all_rewards = []
for it in range(n_iterations):
    # rollout text
    states, actions, rewards, old_log_probs = [], [], [], []
    state = torch.randn(1, 4)
    ep_reward = 0
    for step in range(20):
        with torch.no_grad():
            logits = policy(state)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            old_log_prob = dist.log_prob(action)
        next_state, reward, done = env_step(state.squeeze(), action.item())
        states.append(state.squeeze())
        actions.append(action)
        rewards.append(reward)
        old_log_probs.append(old_log_prob)
        ep_reward += reward
        state = next_state.unsqueeze(0)
    all_rewards.append(ep_reward / 20)

    # advantage Calculation (text: text text)
    advantages = torch.tensor(rewards)
    returns = torch.tensor(rewards)
    values = torch.zeros_like(returns)  # text text (text)

    ppo_update(policy, opt, torch.stack(states), torch.tensor(actions),
               torch.stack(old_log_probs), advantages, returns, values)

print(f"text 10 text Mean Reward: {np.mean(all_rewards[:10]):.4f}")
print(f"text 10 text Mean Reward: {np.mean(all_rewards[-10:]):.4f}")


## 21.6 text Model (RM) Training

RMtext text text text Model. **text Data** (chosen vs rejected)text Training.

**Bradley-Terry Model**:
$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

text (NLL):
$$\mathcal{L}_{\text{RM}} = -\log \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

- $\mathbf{y}_w$: text(chosen) text
- $\mathbf{y}_l$: text(rejected) text
- $r(\mathbf{x}, \mathbf{y})$: RMtext text text

RM text: SFT Modeltext text Outputtext text.


In [ ]:
# RM Training text
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.LSTM(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, 1)  # text text

    def forward(self, x):
        emb = self.emb(x)
        _, (h, _) = self.rnn(emb)
        return self.head(h.squeeze(0))  # (B,)

# toy text Data
n_samples = 100
torch.manual_seed(0)
rm = RewardModel(vocab_size=vocab_size, d_model=32)
opt = torch.optim.AdamW(rm.parameters(), lr=1e-3)

# text chosen/rejected pairs (text)
print("RM Training text:")
losses = []
for step in range(200):
    # text Data: text text
    chosen = torch.randint(0, vocab_size, (8, 16))
    rejected = torch.randint(0, vocab_size, (8, 16))
    # text text: chosentext text IDtext text text text
    # (text RMtext Trainingtext text)
    r_chosen = rm(chosen).squeeze(-1)
    r_rejected = rm(rejected).squeeze(-1)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('RM Loss')
plt.title('Reward Model Training (Bradley-Terry)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch21_rm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"text RM loss: {losses[-1]:.4f} (Theory Minimum ≈ 0.69 = -log(0.5) text)")


## 21.7 KL text — text Modeltext text text textdegreestext

RLHFtext RM text text Modeltext text text text text text (reward hacking).

text: text Model(SFT Model)text text text text KL text.

$$R_{\text{total}} = R_{\text{RM}}(\mathbf{x}, \mathbf{y}) - \beta \, D_{\text{KL}}(\pi_\theta(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

- $\beta$: KL textdegrees (text 0.01~0.5)
- $\pi_{\text{ref}}$: SFT Model (text)
- text KL: $\sum_t \log \frac{\pi_\theta(w_t|...)}{\pi_{\text{ref}}(w_t|...)}$


In [ ]:
# KL text Calculation (text vs text)
def token_kl_div(policy_logits, ref_logits):
    """text KL Divergence."""
    p = F.softmax(policy_logits, dim=-1)
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_q = F.log_softmax(ref_logits, dim=-1)
    return (p * (log_p - log_q)).sum(dim=-1)  # (B, T)

# text
torch.manual_seed(0)
B, T, V = 4, 16, 100
ref_logits = torch.randn(B, T, V)
policy_logits = ref_logits + torch.randn(B, T, V) * 0.5  # text text Policy

kl = token_kl_div(policy_logits, ref_logits)
print(f"KL text shape: {kl.shape}")
print(f"Mean KL: {kl.mean():.4f}")
print(f"Maximum KL: {kl.max():.4f}")
print("\n=> Policytext text text KL Increase. RLHFtext text text text.")


## 21.8 [CPU/GPU Benchmark ⑨] SFT vs RLHF Training Comparison

RLHFtext Model 4text text: text, text, RM, textFunction. text·Time 3~4text.


In [ ]:
# RLHF text text (text)
print("RLHF Trainingtext text Model:")
print("  1. Policy model (Trainingtext)")
print("  2. Reference model (text, KLtext)")
print("  3. Reward model (text)")
print("  4. Value/Critic model (Trainingtext, optional)")
print()

# text: 7B LLaMA RLHF
model_size_gb = 7 * 2 * 4  # 7B params * 2 bytes (FP16) * 4 models? No:
# text: policy (FP32 grad, FP32 opt state)
# AdamW: param + grad + m + v = 4text
# 7B Model FP16: 14GB
# AdamW text FP32: 7B * 8 bytes = 56GB
# policy: 14 + 56 = 70GB
# reference: 14GB
# RM: 14GB
# value: 14 + 56 = 70GB
# text: text 168GB → A100 80GB 2text text

print("7B Model RLHF Memory Estimation:")
print(f"  Policy (FP16 + AdamW FP32): ~70GB")
print(f"  Reference (FP16, text): ~14GB")
print(f"  Reward Model (FP16, text): ~14GB")
print(f"  Value (FP16 + AdamW): ~70GB")
print(f"  text: ~168GB (A100 80GB × 2 text)")
print("\n=> RLHFtext text text. DPO(Ch 22)text text Efficiencytext text.")


## 21.9 RLHFtext text

- **text text(reward hacking)**: RM text text text text text
- **text text(mode collapse)**: text text text text text
- **text**: PPO text text

## 21.10 Key Takeaways

| text | text | text |
|---|---|---|
| text text | $\nabla J = \mathbb{E}[\nabla\log\pi \cdot G]$ | REINFORCE |
| Advantage | $A = G - V$ | text text |
| PPO clip | $\min(rA, \text{clip}(r)A)$ | trust region |
| RM text | $-\log\sigma(r_w - r_l)$ | Bradley-Terry |
| KL text | $R - \beta D_{KL}$ | text text |

## Exercises

1. REINFORCEtext CartPole text toy text Trainingtext.
2. PPOtext $\epsilon = 0.1, 0.2, 0.3$text Comparisontext text text.
3. RM Trainingtext chosentext rejectedtext text text vs text text losstext Comparisontext.
4. KL text $\beta = 0, 0.1, 1.0$text text text text.
5. RLHF text DPOtext text text text text.

> Solutions: `solutions/ch21_solutions.ipynb`
